In [84]:
import pandas as pd
import numpy as np

np.random.seed(42)

# =====================
# Load data
# =====================
train = pd.read_csv("dataset/train.csv")
test = pd.read_csv("dataset/test.csv")

submission = pd.DataFrame(columns=["id", "subtaskID", "answer"])

# ==========================
# CERINTA 1 – materie random
# ==========================
st1 = test[test["liceu"] == "COLEGIUL NATIONAL \"UNIREA\" FOCSANI"]
st1 = st1[st1["materie"] != "LIMBA ROMANA"]
st1 = st1[st1["materie"] != "GENERAL"]
st1 = st1.sort_values("preferinta_materie", ascending=False)

print(st1.iloc[0]["materie"])
print(st1)

submission = pd.concat([submission, pd.DataFrame([{
    "id": 1,
    "subtaskID": 1,
    "answer": st1.iloc[0]["materie"]
}])], ignore_index=True)

MATEMATICA MATE-INFO
           id    an                                            materie  \
10122  130882  2023                               MATEMATICA MATE-INFO   
6792   127552  2023                                            ISTORIE   
3128   123888  2023                                         FIZICA TEO   
9280   130040  2023                  LOGICA, ARGUMENTARE SI COMUNICARE   
6184   126944  2023                               INFORMATICA MI C-C++   
10792  131552  2023                                  MATEMATICA ST-NAT   
5011   125771  2023                                          GEOGRAFIE   
31     120791  2023  ANATOMIE SI FIZIOLOGIE UMANA, GENETICA SI ECOL...   
2268   123028  2023                     CHIMIE ORGANICA TEO NIVEL I-II   
917    121677  2023                       BIOLOGIE VEGETALA SI ANIMALA   
12434  133194  2023                                         PSIHOLOGIE   
2553   123313  2023                                           ECONOMIE   
2679   123439  20

In [85]:
# =====================
# CERINTA 2 – an random
# =====================
st2 = train[train["liceu"] == "COLEGIUL NATIONAL \"UNIREA\" FOCSANI"]
st2 = st2[st2["materie"] == "INFORMATICA MI C-C++"]
st2 = st2.sort_values("preferinta_materie", ascending=False)

submission = pd.concat([submission, pd.DataFrame([{
    "id": 2,
    "subtaskID": 2,
    "answer": st2.iloc[0]["an"]
}])], ignore_index=True)

st2.head()

,id,an,materie,liceu,judet,medie,procent_reusita,numar_candidati,preferinta_materie,anomalie
113940,113941,2022,INFORMATICA MI C-C++,"COLEGIUL NATIONAL ""UNIREA"" FOCSANI",Vrancea,9.60,100.0,36,18.8,0
100860,100861,2021,INFORMATICA MI C-C++,"COLEGIUL NATIONAL ""UNIREA"" FOCSANI",Vrancea,9.51,100.0,34,18.7,0
47589,47590,2017,INFORMATICA MI C-C++,"COLEGIUL NATIONAL ""UNIREA"" FOCSANI",Vrancea,9.02,100.0,32,15.2,0
7366,7367,2014,INFORMATICA MI C-C++,"COLEGIUL NATIONAL ""UNIREA"" FOCSANI",Vrancea,9.13,100.0,29,13.5,0
60777,60778,2018,INFORMATICA MI C-C++,"COLEGIUL NATIONAL ""UNIREA"" FOCSANI",Vrancea,8.94,100.0,27,12.9,0


In [86]:
# =====================
# CERINTA 3
# =====================
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import median_absolute_error
from catboost import CatBoostRegressor

validation_year = train["an"].max()
valid_mask = train["an"] == validation_year
base_train = train.loc[~valid_mask].copy()
base_valid = train.loc[valid_mask].copy()

def add_school_features(stats_source, target):
    school_stats = stats_source.groupby("liceu")["medie"].agg(
        liceu_medie_mean="mean",
        liceu_medie_median="median",
        liceu_medie_std="std",
        liceu_medie_min="min",
        liceu_medie_max="max",
        liceu_count="count",
    ).reset_index()

    subject_stats = stats_source.groupby("materie")["medie"].agg(
        materie_medie_mean="mean",
    ).reset_index()

    county_stats = stats_source.groupby("judet")["medie"].agg(
        judet_medie_mean="mean",
    ).reset_index()

    school_subject_stats = stats_source.groupby(["liceu", "materie"])["medie"].agg(
        liceu_materie_mean="mean",
        liceu_materie_median="median",
        liceu_materie_std="std",
        liceu_materie_count="count",
    ).reset_index()

    county_subject_stats = stats_source.groupby(["judet", "materie"])["medie"].agg(
        judet_materie_mean="mean",
        judet_materie_count="count",
    ).reset_index()

    yearly = stats_source.groupby(["liceu", "materie", "an"], as_index=False)["medie"].mean()
    yearly = yearly.sort_values(["liceu", "materie", "an"])
    yearly["school_materie_roll3_mean"] = yearly.groupby(["liceu", "materie"])["medie"].transform(
        lambda values: values.rolling(3, min_periods=1).mean()
    )
    temporal_stats = yearly.assign(an=yearly["an"] + 1).rename(
        columns={"medie": "school_materie_prev_year_medie"}
    )[[
        "liceu",
        "materie",
        "an",
        "school_materie_prev_year_medie",
        "school_materie_roll3_mean",
    ]]

    global_mean = stats_source["medie"].mean()
    global_median = stats_source["medie"].median()

    featured = target.merge(school_stats, on="liceu", how="left")
    featured = featured.merge(subject_stats, on="materie", how="left")
    featured = featured.merge(county_stats, on="judet", how="left")
    featured = featured.merge(school_subject_stats, on=["liceu", "materie"], how="left")
    featured = featured.merge(county_subject_stats, on=["judet", "materie"], how="left")
    featured = featured.merge(temporal_stats, on=["liceu", "materie", "an"], how="left")

    for col in [
        "liceu_medie_mean",
        "liceu_medie_min",
        "liceu_medie_max",
        "materie_medie_mean",
        "judet_medie_mean",
        "liceu_materie_mean",
        "judet_materie_mean",
        "school_materie_prev_year_medie",
        "school_materie_roll3_mean",
    ]:
        featured[col] = featured[col].fillna(global_mean)

    for col in ["liceu_medie_median", "liceu_materie_median"]:
        featured[col] = featured[col].fillna(global_median)

    for col in [
        "liceu_medie_std",
        "liceu_materie_std",
        "liceu_count",
        "liceu_materie_count",
        "judet_materie_count",
    ]:
        featured[col] = featured[col].fillna(0)

    featured["liceu_vs_judet_mean"] = featured["liceu_medie_mean"] - featured["judet_medie_mean"]
    featured["liceu_materie_vs_materie_mean"] = featured["liceu_materie_mean"] - featured["materie_medie_mean"]
    featured["judet_materie_vs_materie_mean"] = featured["judet_materie_mean"] - featured["materie_medie_mean"]

    return featured.drop(columns=["liceu"])

def encode_features(train_features, other_features):
    train_encoded = pd.get_dummies(train_features, columns=["materie", "judet"], dtype=int)
    other_encoded = pd.get_dummies(other_features, columns=["materie", "judet"], dtype=int)
    train_encoded, other_encoded = train_encoded.align(other_encoded, join="left", axis=1, fill_value=0)
    return train_encoded, other_encoded

# Honest validation: 2022 features are computed only from years before 2022.
base_train_fe = add_school_features(base_train, base_train)
base_valid_fe = add_school_features(base_train, base_valid)
base_train_enc, base_valid_enc = encode_features(base_train_fe, base_valid_fe)

X_train = base_train_enc.drop(columns=["id", "medie", "anomalie"])
X_test = base_valid_enc.drop(columns=["id", "medie", "anomalie"])
y_train = base_train["medie"]
y_test = base_valid["medie"]

# Final submission: train on all available years and compute 2023 features from all training labels.
train_fe = add_school_features(train, train)
test_fe = add_school_features(train, test)
train_enc, test_enc = encode_features(train_fe, test_fe)
X = train_enc.drop(columns=["id", "medie", "anomalie"])
test_enc = test_enc.drop(columns=["id", "medie", "anomalie"], errors="ignore")
y = train["medie"]
print(f"Training on years < {validation_year}; validating on {validation_year}")

cb = CatBoostRegressor(

    iterations=1500,

    learning_rate=0.1,

    depth=10,

    l2_leaf_reg=3,

    loss_function="MAE",

    verbose=100,

    random_seed=42

)

cb.fit(X_train, y_train)

preds = cb.predict(X_test)

mae = median_absolute_error(y_test, preds)

print(f"Validation Median AE: {mae:.4f}")

cb.fit(X, y)
test_preds = cb.predict(test_enc)

test_preds = np.round(test_preds, 2)

submission = pd.concat([submission, pd.DataFrame({
    "id": test["id"],
    "subtaskID": 3,
    "answer": test_preds
})], ignore_index=True)

mae

Training on years < 2022; validating on 2022
0:	learn: 1.3863143	total: 39.8ms	remaining: 59.6s
100:	learn: 0.3510507	total: 1.46s	remaining: 20.2s
200:	learn: 0.3244765	total: 2.86s	remaining: 18.5s
300:	learn: 0.3073276	total: 4.22s	remaining: 16.8s
400:	learn: 0.2933129	total: 5.64s	remaining: 15.5s
500:	learn: 0.2821843	total: 7s	remaining: 14s
600:	learn: 0.2737832	total: 8.35s	remaining: 12.5s
700:	learn: 0.2666523	total: 9.7s	remaining: 11.1s
800:	learn: 0.2590389	total: 11.1s	remaining: 9.65s
900:	learn: 0.2520216	total: 12.4s	remaining: 8.26s
1000:	learn: 0.2460586	total: 13.8s	remaining: 6.87s
1100:	learn: 0.2407328	total: 15.1s	remaining: 5.49s
1200:	learn: 0.2359795	total: 16.5s	remaining: 4.1s
1300:	learn: 0.2319060	total: 17.9s	remaining: 2.73s
1400:	learn: 0.2281205	total: 19.2s	remaining: 1.36s
1499:	learn: 0.2243984	total: 20.6s	remaining: 0us
Validation Median AE: 0.3329
0:	learn: 1.3803134	total: 13.9ms	remaining: 20.8s
100:	learn: 0.3560280	total: 1.43s	remaining: 1

0.33290808074229794

In [87]:
test_preds

array([7.97, 6.33, 9.15, ..., 5.61, 7.88, 8.  ], shape=(13017,))

In [88]:
# =====================
# CERINTA 4
# =====================

anomaly_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
anomaly_model.fit(X, train["anomalie"])
anomaly_preds = anomaly_model.predict(test_enc).astype(int)

submission = pd.concat([submission, pd.DataFrame({
    "id": test["id"],
    "subtaskID": 4,
    "answer": anomaly_preds,
})], ignore_index=True)

In [89]:
# =====================
# Save submission
# =====================
submission_df = pd.DataFrame(submission)
submission_df = submission_df.sort_values(["id", "subtaskID"]).reset_index(drop=True)
submission_df.to_csv("output.csv", index=False)